# Reports Preview

Quick preview of generated reports and audit artifacts.

Steps:
- List recent report files.
- Preview key markdown outputs.
- Summarize report sizes.



In [ ]:
from __future__ import annotations

import json
import os
import sys
import subprocess
from pathlib import Path

# Resolve repo root from the notebook location.
REPO_ROOT = Path.cwd()
for parent in [REPO_ROOT] + list(REPO_ROOT.parents):
    if (parent / 'scripts').exists() and (parent / 'notebooks').exists():
        REPO_ROOT = parent
        break

# Ensure local modules are importable.
sys.path.insert(0, str(REPO_ROOT))
sys.path.insert(0, str(REPO_ROOT / 'src'))

PY = sys.executable

def run(cmd: list[str]) -> None:
    # Run a command from the repo root with PYTHONPATH set.
    env = os.environ.copy()
    env['PYTHONPATH'] = os.pathsep.join([str(REPO_ROOT / 'src'), str(REPO_ROOT)])
    print('$', ' '.join(cmd))
    subprocess.run(cmd, cwd=str(REPO_ROOT), check=True, env=env)

def show_json(rel_path: str) -> None:
    path = REPO_ROOT / rel_path
    if not path.exists():
        print('Missing:', path)
        return
    try:
        data = json.loads(path.read_text(encoding='utf-8'))
    except Exception:
        print(path.read_text(encoding='utf-8', errors='ignore')[:2000])
        return
    print(json.dumps(data, indent=2))

def list_dir(rel_path: str, limit: int = 20) -> None:
    path = REPO_ROOT / rel_path
    if not path.exists():
        print('Missing:', path)
        return
    print(f'\n{rel_path}/')
    for item in sorted(path.iterdir())[:limit]:
        print(' -', item.name)


In [ ]:
from pathlib import Path

reports_dir = REPO_ROOT / 'reports'
summary = {
    'recent_reports': [],
    'sizes': {},
}

if not reports_dir.exists():
    print('Missing:', reports_dir)
else:
    files = [p for p in reports_dir.iterdir() if p.is_file()]
    files.sort(key=lambda p: p.stat().st_mtime, reverse=True)
    summary['recent_reports'] = [str(p.relative_to(REPO_ROOT)) for p in files[:12]]
    for path in files[:12]:
        size_mb = round(path.stat().st_size / 1024**2, 3)
        summary['sizes'][path.name] = size_mb
        print(path.name, size_mb, 'MB')


In [ ]:
# Preview key markdown reports.
keys = ['SYSTEM_SCORECARD.md', 'CLAIM_EVIDENCE.md', 'PROJECT_AUDIT.md', 'SENTIFARGO_AUDIT.md']
for key in keys:
    path = reports_dir / key
    if path.exists():
        print('')
        print(key)
        print(path.read_text(encoding='utf-8', errors='ignore')[:1200])
    else:
        print('Missing:', path)


In [ ]:
# Persist summary for quick reference.
report_dir = REPO_ROOT / 'reports'
report_dir.mkdir(parents=True, exist_ok=True)
summary_path = report_dir / 'evaluation_reports_preview_summary.json'
summary_path.write_text(json.dumps(summary, indent=2))
print('Saved summary to', summary_path)


In [ ]:
# Quick artifact index for verification.
for folder in ['models', 'experiments', 'artifacts', 'runs', 'reports', 'logs']:
    path = REPO_ROOT / folder
    if not path.exists():
        continue
    print(f'\n{folder}/')
    for item in sorted(path.iterdir())[:20]:
        print(' -', item.name)
